# Nemotron 0.87 Training Pipeline

Based on the proven 0.87 approach with:
- Failure mining to identify hard problems
- Synthetic data generation for each problem type
- Curriculum learning (real + synthetic, oversample hard)
- QLoRA warmstart from huikang adapter
- Short training (240 steps)

Expected runtime: ~1-2 hours on Kaggle GPU

## 1. Environment Setup & Imports

In [ ]:
import subprocess, sys, os, warnings, tempfile, zipfile
warnings.filterwarnings("ignore")

def install_mamba():
    try:
        import mamba_ssm, causal_conv1d
        print("mamba-ssm already available")
        return
    except ImportError:
        pass

    for wd in ['/kaggle/input/nemotron-packages',
               '/kaggle/input/datasets/mayukh18/nemotron-packages']:
        if os.path.isdir(wd):
            for f in sorted(os.listdir(wd)):
                if f.endswith('.whl') and ('mamba_ssm' in f or 'causal_conv1d' in f):
                    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', os.path.join(wd, f)])
                    print(f'Installed: {f}')

    try:
        import mamba_ssm, causal_conv1d
        print("mamba-ssm OK (mounted)")
        return
    except ImportError:
        pass

    for ds in ['mayukh18/nemotron-packages']:
        try:
            print(f"Downloading {ds} via Kaggle CLI...")
            tmpdir = tempfile.mkdtemp()
            subprocess.run(['kaggle', 'datasets', 'download', ds, '-p', tmpdir, '--unzip'],
                          check=False, capture_output=True, timeout=60)
            for f in os.listdir(tmpdir):
                fp = os.path.join(tmpdir, f)
                if f.endswith('.zip'):
                    with zipfile.ZipFile(fp) as zf:
                        zf.extractall(tmpdir)
            for f in os.listdir(tmpdir):
                fp = os.path.join(tmpdir, f)
                if f.endswith('.whl') and ('mamba_ssm' in f or 'causal_conv1d' in f):
                    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', fp])
                    print(f'Installed: {f}')
        except Exception as e:
            print(f"  Failed: {e}")

    try:
        import mamba_ssm, causal_conv1d
        print("mamba-ssm OK")
        return
    except ImportError:
        raise RuntimeError("Cannot install mamba-ssm.")

install_mamba()

for pkg in ['bitsandbytes', 'accelerate', 'peft']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Dependencies ready")

from pathlib import Path
import gc, json, math, random, re, shutil, string
import numpy as np
import pandas as pd
import torch

SEED = 3407
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def find_path(*candidates):
    for p in candidates:
        pp = Path(p)
        if pp.exists():
            return pp
    return Path(candidates[0])

COMP_DIR = find_path(
    '/kaggle/input/nvidia-nemotron-model-reasoning-challenge',
    '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge',
)
TRAIN_CSV = COMP_DIR / 'train.csv'
print(f'Competition: {COMP_DIR.exists()}')

BASE_MODEL_PATH = str(find_path(
    '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1',
))
print(f'Base model: {Path(BASE_MODEL_PATH).exists()}')

WARMSTART_ADAPTER_DIR = find_path(
    '/kaggle/input/models/huikang/nemotron-adapter/transformers/default/27',
    '/kaggle/input/models/huikang/nemotron-adapter/transformers/default/1',
)
HAS_WARMSTART = WARMSTART_ADAPTER_DIR.exists()
print(f'Huikang adapter: {HAS_WARMSTART}')

if not Path(BASE_MODEL_PATH).exists():
    raise RuntimeError("Base model not mounted.")

REFERENCE_ZIP = Path('/kaggle/input/notebooks/huikang/nvidia-nemotron-all-linear/reference/submission.zip')
OUTPUT_ROOT = Path('/kaggle/working/pipeline')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER_DIR = OUTPUT_ROOT / 'trained_adapter'
SUBMISSION_ZIP = Path('/kaggle/working/submission.zip')

print("Setup complete")

## 2. Load Training Data & Run Failure Mining

In [ ]:
# Better bucket classification using keyword matching
ALPHABET = string.ascii_lowercase
WORD_BANK = ['stone','cable','garden','planet','window','rocket','silver','harbor','market','forest','magnet','bridge','castle','winter','summer','spring','autumn','tower','valley','island']

def prompt_bucket(text):
    t = str(text).lower()
    if 'bit manipulation' in t or 'bit shift' in t or 'binary numbers' in t:
        return 'bit_like'
    if 'encrypt' in t or 'decrypt' in t or 'cipher' in t:
        return 'cipher_like'
    if 'numeral' in t or 'roman' in t:
        return 'numeral_like'
    if 'unit conversion' in t or ('convert' in t and 'unit' in t):
        return 'unit_like'
    if 'equation' in t or 'algebra' in t or ('solve' in t and 'x' in t):
        return 'equation_like'
    if 'gravity' in t or 'physics' in t or 'planet' in t:
        return 'gravity_like'
    return 'other'

def prompt_fingerprint(text):
    t = str(text).lower()
    length_bucket = min(5, len(text) // 200)
    nums = len(re.findall(r'-?\d+', t))
    return f'l{length_bucket}|n{min(nums,12)}'

# Load data
train_df = pd.read_csv(TRAIN_CSV)
train_df['bucket'] = train_df['prompt'].apply(prompt_bucket)
train_df['fingerprint'] = train_df['prompt'].apply(prompt_fingerprint)
train_df['prompt_len'] = train_df['prompt'].str.len()

bucket_report = train_df.groupby('bucket').agg(
    examples=('id','count'),
    avg_prompt_len=('prompt_len','mean'),
    unique_fingerprints=('fingerprint','nunique')
).reset_index()

bucket_report['failure_weight'] = bucket_report['examples'] * 0.4 + bucket_report['unique_fingerprints'] * 3.0
bucket_report = bucket_report.sort_values('failure_weight', ascending=False).reset_index(drop=True)

# Score each problem for difficulty
weight_map = dict(zip(bucket_report['bucket'], bucket_report['failure_weight']))
train_df['hard_score'] = train_df['bucket'].map(weight_map).fillna(1.0) + train_df['prompt_len'] / 400.0
failure_report = train_df.sort_values(['hard_score','prompt_len'], ascending=[False,False]).reset_index(drop=True)

print("Bucket report:")
print(bucket_report.to_string())
print(f"\nTotal training samples: {len(train_df)}")
print(f"Buckets found: {train_df['bucket'].nunique()}")

## 3. Generate Synthetic Training Data

In [ ]:
def format_examples(title, examples, target_input):
    lines = [title, '', 'Infer the hidden rule from the examples and solve the target case.', '']
    for i, (inp, out) in enumerate(examples, 1):
        lines += [f'Example {i}:', f'Input: {inp}', f'Output: {out}', '']
    lines += ['Target:', f'Input: {target_input}', 'Output:']
    return '\n'.join(lines)

RANDOM_SYNTHETIC_EXAMPLES = 12000

def gen_bit_like(rng):
    shift = rng.choice([1,2])
    rules = [
        ('invert_bits', lambda s: ''.join('1' if c == '0' else '0' for c in s), 'The rule flips every bit independently.'),
        ('reverse_bits', lambda s: s[::-1], 'The rule reverses the bit string.'),
        (f'rotate_left_{shift}', lambda s,k=shift: s[k:] + s[:k], f'The rule rotates the bit string left by {shift} positions.'),
    ]
    name, fn, expl = rng.choice(rules)
    bit_len = rng.choice([6,7,8])
    vals = []
    while len(vals) < 4:
        s = ''.join(rng.choice('01') for _ in range(bit_len))
        if s not in vals:
            vals.append(s)
    examples = [(s, fn(s)) for s in vals[:3]]
    target = vals[3]
    answer = fn(target)
    return {'bucket':'bit_like', 'prompt':format_examples('Bit transformation puzzle', examples, target), 'answer':answer, 'trace':f'{expl} Applying it to {target} gives the answer. Final answer: \\boxed{{{answer}}}.'}

def gen_cipher_like(rng):
    shift = rng.choice([1,2,3,4])
    rules = [
        ('reverse_word', lambda w: w[::-1], 'The rule reverses the characters in each word.'),
        (f'caesar_plus_{shift}', lambda w,k=shift: ''.join(ALPHABET[(ALPHABET.index(ch)+k)%26] for ch in w), f'The rule shifts every letter forward by {shift}.'),
        ('atbash', lambda w: ''.join(ALPHABET[25-ALPHABET.index(ch)] for ch in w), 'The rule uses Atbash cipher: a-\u003ez, b-\u003ey, and so on.'),
    ]
    name, fn, expl = rng.choice(rules)
    words = rng.sample(WORD_BANK, min(4, len(WORD_BANK)))
    examples = [(w, fn(w)) for w in words[:3]]
    target = words[3] if len(words) >= 4 else words[-1]
    answer = fn(target)
    return {'bucket':'cipher_like', 'prompt':format_examples('Word cipher puzzle', examples, target), 'answer':answer, 'trace':f'{expl} Applying it to \"{target}\" gives the answer. Final answer: \\boxed{{{answer}}}.'}

def gen_unit_like(rng):
    rules = [('km_to_m',1000,'km','m'),('kg_to_g',1000,'kg','g'),('hour_to_min',60,'hour','minute'),('day_to_hour',24,'day','hour'),('m_to_cm',100,'m','cm'),('week_to_day',7,'week','day')]
    name, factor, src, dst = rng.choice(rules)
    vals = rng.sample(range(2,25),4)
    examples = [(f'{v} {src}', str(v*factor)) for v in vals[:3]]
    target = vals[3]
    answer = str(target*factor)
    return {'bucket':'unit_like', 'prompt':format_examples(f'Unit conversion puzzle ({src} to {dst})', examples, f'{target} {src}'), 'answer':answer, 'trace':f'The rule converts from {src} to {dst} using a factor of {factor}. The target becomes {answer}. Final answer: \\boxed{{{answer}}}.'}

def gen_numeral_like(rng):
    rules = [
        ('decimal_to_binary', lambda n: format(n, 'b'), lambda n: f'{n} (base 10)', 'The rule converts each decimal number into binary.'),
        ('decimal_to_hex', lambda n: format(n, 'x'), lambda n: f'{n} (base 10)', 'The rule converts each decimal number into hexadecimal.'),
    ]
    name, out_fn, in_fn, expl = rng.choice(rules)
    nums = rng.sample(range(6,40),4)
    examples = [(in_fn(n), out_fn(n)) for n in nums[:3]]
    target = nums[3]
    answer = out_fn(target)
    return {'bucket':'numeral_like', 'prompt':format_examples('Numeral system puzzle', examples, in_fn(target)), 'answer':answer, 'trace':f'{expl} Applying it to the target gives the answer. Final answer: \\boxed{{{answer}}}.'}

def gen_equation_like(rng):
    a = rng.choice([1,2,3,4,5,6])
    def sample_eq():
        x = rng.choice(range(-8,9))
        b = rng.choice(range(-9,10))
        c = a * x + b
        eq = f'{a}x + {b} = {c}' if b >= 0 else f'{a}x - {abs(b)} = {c}'
        return eq, str(x)
    examples = [sample_eq() for _ in range(3)]
    target = sample_eq()
    return {'bucket':'equation_like', 'prompt':format_examples('Linear equation puzzle', examples, target[0]), 'answer':target[1], 'trace':f'Each output is the value of x that satisfies the equation. For the target equation, x = {target[1]}. Final answer: \\boxed{{{target[1]}}}.'}

GENERATORS = {'bit_like':gen_bit_like, 'cipher_like':gen_cipher_like, 'unit_like':gen_unit_like, 'numeral_like':gen_numeral_like, 'equation_like':gen_equation_like}
rng = random.Random(SEED)

supported = bucket_report[bucket_report['bucket'].isin(GENERATORS.keys())].copy()
supported['allocation_weight'] = supported['failure_weight'].clip(lower=1.0)
supported['allocation_weight'] = supported['allocation_weight'] / supported['allocation_weight'].sum()
supported['synth_count'] = (supported['allocation_weight'] * RANDOM_SYNTHETIC_EXAMPLES).round().astype(int)

synth_rows = []
for _, row in supported.iterrows():
    g = GENERATORS[row['bucket']]
    for _ in range(int(row['synth_count'])):
        synth_rows.append(g(rng))

synth_df = pd.DataFrame(synth_rows)
print(f"Generated {len(synth_df)} synthetic examples")
print(supported[['bucket','synth_count','allocation_weight']].to_string())
print(f"\nSample synthetic data:")
print(synth_df.head(2)[['bucket','answer','trace']])

## 4. Build Curriculum Dataset

In [ ]:
REAL_HARD_OVERSAMPLE = 4
TRACELESS_SHORT_REASONING = True

def build_user_prompt(prompt):
    return prompt + '\nPlease put your final answer inside \\boxed{}.'

def make_record(record_id, bucket, source, prompt, response, meta=None):
    return {
        'id': record_id,
        'bucket': bucket,
        'source': source,
        'messages': [
            {'role': 'user', 'content': build_user_prompt(prompt)},
            {'role': 'assistant', 'content': response},
        ],
        'meta': meta or {},
    }

def real_short_trace(answer):
    if TRACELESS_SHORT_REASONING:
        return 'I identify one rule that matches all examples, verify that the same rule is consistent across the prompt, then apply it to the target. Final answer: \\boxed{' + str(answer) + '}.'
    return '\\boxed{' + str(answer) + '}'

# Build curriculum
hard_threshold = failure_report['hard_score'].quantile(0.8)
real_rows = []

for _, row in failure_report.iterrows():
    # Oversample hard problems
    oversample = REAL_HARD_OVERSAMPLE if row['hard_score'] >= hard_threshold else 1
    for replica_idx in range(oversample):
        real_rows.append(make_record(
            f"{row['id']}:answer:{replica_idx}", row['bucket'], 'real_answer_only',
            row['prompt'], f"\\boxed{{{row['answer']}}}",
            {'hard_score': float(row['hard_score']), 'fingerprint': row['fingerprint']}
        ))
    # Add short reasoning trace for each problem
    real_rows.append(make_record(
        f"{row['id']}:trace:0", row['bucket'], 'real_short_trace',
        row['prompt'], real_short_trace(row['answer']),
        {'hard_score': float(row['hard_score']), 'fingerprint': row['fingerprint']}
    ))

# Add synthetic records with full traces
synth_records = [
    make_record(f'synth:{i}', row['bucket'], 'synthetic_trace', row['prompt'], row['trace'])
    for i, row in synth_df.iterrows()
]

# Combine and shuffle
final_records = real_rows + synth_records
rng.shuffle(final_records)

print(f"Real records: {len(real_rows)}")
print(f"Synthetic records: {len(synth_records)}")
print(f"Total curriculum: {len(final_records)}")
print(f"Hard threshold score: {hard_threshold:.2f}")
print(f"Hard problems (oversampled): {len(failure_report[failure_report['hard_score'] >= hard_threshold])}")

## 5. Tokenize Curriculum

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset as TorchDataset

MAX_LENGTH = 6144

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_messages(messages, add_generation_prompt=False):
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)
    except Exception:
        text = ''
        for message in messages:
            text += f"{message['role'].upper()}:\n{message['content']}\n\n"
        if add_generation_prompt:
            text += 'ASSISTANT:\n'
        return text

print("Tokenizing curriculum...")
tokenized_rows = []
for rec in final_records:
    prompt_messages = rec['messages'][:1]
    full_messages = rec['messages']
    prompt_text = render_messages(prompt_messages, add_generation_prompt=True)
    full_text = render_messages(full_messages, add_generation_prompt=False)

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)['input_ids']
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)['input_ids']

    labels = ([-100] * len(prompt_ids) + full_ids[len(prompt_ids):])[:len(full_ids)]
    tokenized_rows.append({'input_ids': full_ids, 'attention_mask': [1] * len(full_ids), 'labels': labels})

class ListDataset(TorchDataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx): return self.rows[idx]

class DataCollator:
    def __call__(self, features):
        max_len = max(len(x['input_ids']) for x in features)
        input_ids, attention_mask, labels = [], [], []
        for row in features:
            pad = max_len - len(row['input_ids'])
            input_ids.append(row['input_ids'] + [tokenizer.pad_token_id] * pad)
            attention_mask.append(row['attention_mask'] + [0] * pad)
            labels.append(row['labels'] + [-100] * pad)
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

print(f"Tokenized {len(tokenized_rows)} records")
avg_len = sum(len(r['input_ids']) for r in tokenized_rows) / len(tokenized_rows)
print(f"Average tokenized length: {avg_len:.0f}")

## 6. Load Model with QLoRA & Warmstart Adapter

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

MAX_STEPS = 240
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 16
LORA_RANK = 32

print("Loading model with 8-bit (balanced dual-GPU)...")
gpu_count = torch.cuda.device_count()
print(f"GPU count: {gpu_count}")
for i in range(gpu_count):
    print(f"  GPU {i}: {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

max_mem = {i: f"{int(torch.cuda.get_device_properties(i).total_memory/1e9 * 0.48)}GiB" for i in range(gpu_count)}
max_mem['cpu'] = '48GiB'
print(f"max_memory: {max_mem}")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    trust_remote_code=True,
    quantization_config=quant_config,
    device_map='balanced',
    max_memory=max_mem,
)
print("Model loaded OK")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

if HAS_WARMSTART:
    print("Loading warmstart adapter...")
    model = PeftModel.from_pretrained(model, WARMSTART_ADAPTER_DIR, is_trainable=True)
else:
    print("No warmstart - fresh LoRA init")
    model = get_peft_model(model, LoraConfig(
        r=LORA_RANK, lora_alpha=32, target_modules="all-linear",
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
    ))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,}")
for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.1f}GB / {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")

## 7. Train

In [ ]:
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(FINAL_ADAPTER_DIR),
        overwrite_output_dir=True,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        max_steps=MAX_STEPS,
        logging_steps=5,
        save_steps=60,
        save_total_limit=2,
        bf16=torch.cuda.is_available(),
        report_to=[],
        remove_unused_columns=False,
        optim='paged_adamw_8bit',
        gradient_checkpointing=True,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
    ),
    train_dataset=ListDataset(tokenized_rows),
    data_collator=DataCollator(),
    tokenizer=tokenizer,
)

print(f"Starting training: {MAX_STEPS} steps, LR={LEARNING_RATE}, batch={PER_DEVICE_BATCH_SIZE}x{GRAD_ACCUM}={PER_DEVICE_BATCH_SIZE*GRAD_ACCUM}")
trainer.train()

# Save adapter
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print(f"Adapter saved to {FINAL_ADAPTER_DIR}")
for f in sorted(FINAL_ADAPTER_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size / 1024 / 1024:.1f} MB")

# Cleanup
del model, trainer
gc.collect()
torch.cuda.empty_cache()

## 8. Package submission.zip

In [ ]:
import zipfile

# Extract reference config if available
reference_adapter_config = None
if REFERENCE_ZIP.exists():
    reference_dir = OUTPUT_ROOT / 'reference_adapter'
    shutil.rmtree(reference_dir, ignore_errors=True)
    reference_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(REFERENCE_ZIP, 'r') as zf:
        zf.extractall(reference_dir)
    cfg_path = reference_dir / 'adapter_config.json'
    if cfg_path.exists():
        reference_adapter_config = json.loads(cfg_path.read_text())
        print("Reference config loaded")

# Package adapter
package_dir = OUTPUT_ROOT / 'package_adapter'
shutil.rmtree(package_dir, ignore_errors=True)
shutil.copytree(FINAL_ADAPTER_DIR, package_dir, dirs_exist_ok=True)

# Align config
config_path = package_dir / 'adapter_config.json'
config = json.loads(config_path.read_text())
config['inference_mode'] = True
config['base_model_name_or_path'] = str(BASE_MODEL_PATH)

if reference_adapter_config is not None:
    for key in ['alora_invocation_tokens','arrow_config','ensure_weight_tying','peft_version','qalora_group_size','target_parameters','use_qalora']:
        if key in reference_adapter_config:
            config[key] = reference_adapter_config[key]

config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(f"Config: rank={config.get('r')}, targets={config.get('target_modules')}")

# Create submission zip
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in ['adapter_config.json', 'adapter_model.safetensors']:
        fpath = package_dir / name
        if fpath.exists():
            zf.write(fpath, arcname=name)
        else:
            raise FileNotFoundError(f"Missing {name} in package")

size_mb = SUBMISSION_ZIP.stat().st_size / 1024 / 1024
print(f"\nsubmission.zip created: {size_mb:.1f} MB")
print("Ready for submission!")

## Key Improvements Over Baseline (0.86 -> 0.87)

1. **Synthetic Data**: ~12,000 synthetic examples across 5 problem types with full CoT traces
2. **Failure Mining**: Identifies hard problem types and individual difficult examples
3. **Curriculum Learning**: Hard problems are oversampled 4x + each problem gets a short reasoning trace
4. **QLoRA Warmstart**: Starts from huikang's pretrained adapter, uses 4-bit quantization to fit in memory
5. **Config Alignment**: Reference submission zip provides missing config keys for vLLM compatibility

This approach achieved **0.87 on the public LB**.